# EEE 457 Transmissão de Energia Elétrica
## Escola Politécnica
## Universidade Federal do Rio de Janeiro
### Antonio C. S. Lima
Outubro 2025
### Cálculo de parâmetros unitários em linhas de transmissão aéreas 

In [1]:
# carrega as bibliotecas

import numpy as np
from numpy import pi, sqrt, log, exp, diag, eye, kron
from numpy.linalg import inv, eig
from scipy.special import iv, kv
import matplotlib.pyplot as plt
import pandas as pd
from typing import Iterable, Tuple, Optional, Union
import warnings 
import os
import line_cable_param as lcp

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "sans-serif",
    "font.size": 14,
    "font.sans-serif": "Helvetica",
})

In [2]:
# constantes importantes
µ0 = 4e-7 * pi  # permeabilidade do vácuo (H/m)
ε0 = 8.854187817e-12  # permissividade do vácuo (F/m)
c0 = 1 / sqrt(µ0 * ε0)  # velocidade da luz no vácuo (m/s)
η0 = sqrt(µ0 / ε0)  # impedância característica do vácuo (Ohms)
freq = 60  # frequência de operação (Hz)
omega = 2 * pi * freq  # frequência angular (rad/s)
a = np.exp( 2j * pi / 3)  #  1/_ 120 graus para o calculo dos parâmetros de sequência 

In [3]:
# primeira configuração 
# Circuito 345 kV (345CSH1) com dois condutores Tern
s = 0.4572  # m
f = 19.1  # m flecha dos condutores
fpr = 15.24  # m flecha dos para-raios
xc = np.array([-8.5 - s/2,  -8.5 + s/2, - s/2, s/2, 8.5 - s/2, 8.5 + s/2, -6.25, 6.25] ) # m
yc = np.array([28.4, 28.4, 29.25, 29.25, 28.4, 28.4, 35.9, 35.9 ])  # m
fc = np.concat( [6 * [f], 2* [fpr] ] )  # m flechas dos condutores 
yc = np.array(yc) - 2/3 * fc  # altura media dos condutores 
rho_s = 1e3                   # restividade do solo em  (Ohm . m)
ri = (6.75e-3)/2              # raio interno dos condutores de fase (m)
rf = (27.01e-3)/2             # raio externo dos condutores de fase (m)
rdc = 0.0734e-3               # resistência DC dos condutores de fase (Ohm/m)
nb = 2
npr = 2
rpr = 4.57e-3               # raio dos para-raios (m) considerando 3/8" EHS
rdc_pr =  4.190e-3           # resistência DC dos para-raios (Ohm/m)

In [4]:
# czyl_overhead_bundled(omega, x, y, sigma_s, rdc, rf, rint, npr, rdcpr, rpr, nb):
Z, Y = lcp.czyl_overhead_bundled(omega = omega, 
                                 x = xc,
                                 y = yc,
                                 sigma_s = 1/rho_s,
                                 rdc = rdc, 
                                 rf = rf,
                                 rint = ri, 
                                 npr = npr, 
                                 rdcpr = rdc_pr, 
                                 rpr = rpr,
                                 nb = nb)

In [5]:
print("Impedance Matrix (Ohm/km):")
print(Z * 1000)

Impedance Matrix (Ohm/km):
[[0.13502452+0.75447764j 0.09921839+0.39127967j 0.09741758+0.34096048j]
 [0.09921839+0.39127967j 0.13821513+0.75151352j 0.09921839+0.39127967j]
 [0.09741758+0.34096048j 0.09921839+0.39127967j 0.13502452+0.75447764j]]


In [6]:
print("Admitance (S/km):")
print(Y * 1000)

Admitance (S/km):
[[3.e-09+3.80677820e-06j 0.e+00-6.77018964e-07j 0.e+00-2.13597971e-07j]
 [0.e+00-6.77018964e-07j 3.e-09+3.93105230e-06j 0.e+00-6.77018964e-07j]
 [0.e+00-2.13597971e-07j 0.e+00-6.77018964e-07j 3.e-09+3.80677820e-06j]]


In [7]:
# matriz de Fortescue
A = np.array([[1, 1, 1],[1, a**2, a],[1, a, a**2]])
A_inv = np.linalg.inv(A)
# impedancias de sequência
Z_seq = A_inv @ Z @ A
# admitancias de sequência
Y_seq = A_inv @ Y @ A

In [8]:
print(Z_seq * 1000)

[[ 0.33332429+1.50250281j  0.01283832-0.00933341j -0.01450214-0.00645161j]
 [-0.01450214-0.00645161j  0.03746994+0.37898299j -0.02983897+0.01738573j]
 [ 0.01283832-0.00933341j  0.02997597+0.01714844j  0.03746994+0.37898299j]]


In [9]:
print(Y_seq * 1000)

[[ 3.00000000e-09+2.80311230e-06j -9.79032757e-08+5.65244826e-08j
   9.79032757e-08+5.65244826e-08j]
 [ 9.79032757e-08+5.65244826e-08j  3.00000000e-09+4.37074820e-06j
   3.03431077e-07-1.75186014e-07j]
 [-9.79032757e-08+5.65244826e-08j -3.03431077e-07-1.75186014e-07j
   3.00000000e-09+4.37074820e-06j]]


In [10]:
z012= np.diag(Z_seq)
y012= np.diag(Y_seq)

In [11]:
y1 = y012[1] * 1000  # S/m para S/km
print(y1 * 1e6)  # µS/km

(0.0029999999999996796+4.370748201808891j)


In [12]:
z1  = z012[1] * 1000  # Ohm/m para Ohm/km
print(z1)  # Ohm/km

(0.037469938395084515+0.3789829947153607j)


In [13]:
zc = np.sqrt(z012[1]/y012[1])
print("Impedância característica (Ohm):", np.real(zc))

Impedância característica (Ohm): 294.82748782026886


In [14]:
# potencia natural do circuito
Vn = 345
Pn = Vn**2 / np.real(zc)

print(Pn)

403.7106610376824
